# Depth Anything Evaluation Notebook

This notebook evaluates either **Depth-Anything V2** or **Depth Anything 3** on the same validation split used by the training notebook.

It supports two modes:

1. `WEIGHTS_SOURCE = "pretrained"`: evaluate the model out of the box.
2. `WEIGHTS_SOURCE = "checkpoint"`: load a checkpoint produced by `da_train` and evaluate the trained head.


# Setup

In [7]:
# Optional install if packages are missing in the current environment
# %pip install torch numpy pandas tqdm huggingface-hub

# required for DA2 evaluation
# %pip install transformers

# required by DA3 evaluation
# %pip install xformers torchvision
# %pip install git+https://github.com/ByteDance-Seed/Depth-Anything-3.git

In [8]:
from pathlib import Path
from torch.utils.data import DataLoader, Subset
from dataset import TrainDataset, make_or_load_split
from huggingface_hub import hf_hub_download

import json
import os

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm

## Configuration
Change `MODEL_FAMILY` and `WEIGHTS_SOURCE` here to switch between 
`DA2`/`DA3` and pretrained/checkpoint evaluation.

Only one checkpoint should be active at a time and must match the chosen `MODEL_FAMILY`.

The training data to evaluate the model on should be in the *data/train* directory and must consist of RGB images, each paired with a depth map saved as a numpy array

In [9]:
# ========================================
# Configuration cell: edit only this cell
# ========================================

# Choose the model family: "DA2" or "DA3"
MODEL_FAMILY = "DA2"

# Choose which weights to evaluate.
# "pretrained"  -> evaluate the original pretrained model
# "checkpoint"  -> load trained head weights from CHECKPOINT_PATH
WEIGHTS_SOURCE = "checkpoint"

# If WEIGHTS_SOURCE is "checkpoint", uncomment the checkpoint to load.
# CHECKPOINT = 'DA2_trained.pt'
CHECKPOINT = 'DA2_finetuned.pt'
# CHECKPOINT = 'DA3_trained.pt'
# CHECKPOINT = 'DA3_finetuned.pt'

# path to training data
# The data is expected to have the same format as in the kaggle competition
PROJECT_ROOT = Path("..")
TRAIN_DIR = PROJECT_ROOT / "data/train"

Checkpoints are downloaded from Hugging Face.

The validation split is stored on locally for easy re-use and so that training and evaluation use the exact same samples.

`MIN_DEPTH` and `MAX_DEPTH` define the valid metric range.
Pixels outside this interval are ignored when computing evaluation metrics.

In [ ]:
HF_REPO_ID = "ragerber13/depth-anything-custom-heads"
HF_CHECKPOINT_SUBFOLDER = "checkpoints"

VAL_FRAC = 0.10
SPLIT_SEED = 42

SPLIT_DIR = PROJECT_ROOT / "splits"
os.makedirs(SPLIT_DIR, exist_ok=True)
SPLIT_PATH = SPLIT_DIR / f"train_val_split_seed{SPLIT_SEED}_val{int(VAL_FRAC * 100)}.json"

RESULTS_DIR = PROJECT_ROOT / "evaluation_results"
os.makedirs(RESULTS_DIR, exist_ok=True)

BATCH_SIZE = 4
NUM_WORKERS = 4

MIN_DEPTH = 1e-3
MAX_DEPTH = 80.0

In [11]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Load Model

This section constructs the selected Depth Anything model and optionally replaces its prediction head with a trained checkpoint.

In [ ]:
def get_hf_checkpoint_path():
    ''' 
    Build the checkpoint filename expected inside the Hugging Face repository.
    '''
    filename = f"{HF_CHECKPOINT_SUBFOLDER}/{CHECKPOINT}"

    return hf_hub_download(HF_REPO_ID, filename)


def load_head_checkpoint(model, checkpoint_path):
    ''' 
    Download the selected checkpoint file from Hugging Face and update the DPT head weights.
    '''
    checkpoint = torch.load(checkpoint_path, map_location=device)

    # Load only the trained depth head, not the entire model.
    state_dict = checkpoint["model_head"]

    # DA2 and DA3 have their prediction heads at different locations,
    # so the correct attribute has to be selected based on MODEL_FAMILY.
    if MODEL_FAMILY == "DA2":
        model.head.load_state_dict(state_dict)
    else:
        model.model.head.load_state_dict(state_dict)

    print(f"Loaded checkpoint: {checkpoint_path}")


# Load the base pretrained model
if MODEL_FAMILY == "DA2":
    from transformers import AutoModelForDepthEstimation

    model = AutoModelForDepthEstimation.from_pretrained("depth-anything/Depth-Anything-V2-Metric-Outdoor-Large-hf").to(device)

elif MODEL_FAMILY == "DA3":
    from depth_anything_3.api import DepthAnything3

    model = DepthAnything3.from_pretrained("depth-anything/da3metric-large").to(device)

else:
    raise ValueError('Invalid MODEL_FAMILY. Accepted values are "DA2" or "DA3".')

#  Optionally overwrite the head weights.
if WEIGHTS_SOURCE == "checkpoint":
    CHECKPOINT_PATH = get_hf_checkpoint_path()
    load_head_checkpoint(model, CHECKPOINT_PATH)

elif WEIGHTS_SOURCE != "pretrained":
    raise ValueError('Invalid WEIGHTS_SOURCE. Accepted values are "pretrained" or "checkpoint".')


if MODEL_FAMILY == "DA2":
    model.eval()
else:
    model.model.eval()

Loading weights:   0%|          | 0/503 [00:00<?, ?it/s]

Loaded checkpoint: /home/raphael/.cache/huggingface/hub/models--ragerber13--depth-anything-custom-heads/snapshots/42d86f6174decc51be6a7bff3e8f3a5e9b7eccbd/checkpoints/DA2_finetuned.pt


## Load Data

This section recreates the validation subset using the same saved train/validation split that was used during training.

In [13]:
dataset = TrainDataset(str(TRAIN_DIR))

train_indices, val_indices = make_or_load_split(
    train_dataset=dataset,
    split_path=SPLIT_PATH,
    split_seed=SPLIT_SEED,
    val_frac=VAL_FRAC,
    train_dir=TRAIN_DIR,
)

val_set = Subset(dataset, val_indices)

val_loader = DataLoader(
    val_set,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

Loaded split from ../splits/train_val_split_seed42_val10.json


# Evaluation

### Prediction Helpers

These helpers make sure that both DA2 and DA3 predictions have the same shape as the target depth maps before the metrics are computed.


In [ ]:
def ensure_bchw(x):
    '''
    Convert [B, H, W] to [B, 1, H, W].
    '''
    if x.ndim == 3:
        x = x.unsqueeze(1)

    # Remove extra singleton dimensions that appear in DA3 outputs.
    while x.ndim > 4 and x.shape[1] == 1:
        x = x.squeeze(1)

    return x


def resize_pred_to_target(pred, target):
    ''' 
    Resize predictions to the target resolution before computing pixelwise metrics.
    '''
    pred = ensure_bchw(pred)
    target = ensure_bchw(target)

    if pred.shape[-2:] != target.shape[-2:]:
        # Bilinear interpolation is used because pred is a continuous value map.
        pred = F.interpolate(
            pred,
            size=target.shape[-2:],
            mode="bilinear",
            align_corners=False,
        )

    return pred, target

### Evaluation Metrics

All metrics are computed only for valid pixels, i.e., no NaNs, infinities, non-positive predictions, and target depths outside the configured range

- `mean_si_RMSE`: mean of per-image si-RMSE values
- `AbsRel`: mean absolute relative error over all valid pixels
- `Delta1`: fraction of valid pixels where the prediction is within a factor of `1.25`


In [ ]:
def valid_depth_mask(pred, target):
    ''' 
    Returns a mask of all valid pixels (both prediction and target are finite,
    the prediction is positive, and the target lies inside the depth range).
    '''
    return (
        torch.isfinite(pred)
        & torch.isfinite(target)
        & (pred > 1e-12)
        & (target >= MIN_DEPTH)
        & (target <= MAX_DEPTH)
    )


def si_rmse(pred_valid, target_valid, eps=1e-12):
    ''' 
    Scale-invariant RMSE is computed in log-depth space.
    Subtracting the squared mean log difference removes global scale bias.
    '''
    pred_valid = pred_valid.clamp_min(eps)
    target_valid = target_valid.clamp_min(eps)

    log_diff = torch.log(pred_valid) - torch.log(target_valid)

    return torch.sqrt(
        torch.mean(log_diff ** 2) - torch.mean(log_diff) ** 2
    )


@torch.no_grad()
def evaluate(model, dataloader, device):
    if MODEL_FAMILY == "DA2":
        model.eval()
    else:
        model.model.eval()

    per_image_sirmse = []

    absrel_sum = 0.0
    delta1_sum = 0.0
    valid_pixel_count = 0

    progress_bar = tqdm(
        enumerate(dataloader),
        total=len(dataloader),
        desc="Evaluating",
        unit="batch",
    )

    for step, batch in progress_bar:
        images = batch["image"].to(device)
        target = batch["depth"].to(device).float()

        if MODEL_FAMILY == "DA2":
            pred = ensure_bchw(model(images).predicted_depth)
        else:
            images = images.unsqueeze(1)
            pred = ensure_bchw(model.model(images).depth)

        pred, target = resize_pred_to_target(pred, target)

        mask = valid_depth_mask(pred, target)

        # Store one si-RMSE value per image, then average across images at the end.
        for b in range(pred.shape[0]):
            mask_b = mask[b]

            pred_b = pred[b][mask_b]
            target_b = target[b][mask_b]

            sirmse_b = si_rmse(pred_b, target_b)
            per_image_sirmse.append(sirmse_b.detach().cpu())


        # Clamp values before divisions and logarithms to avoid numerical instability.
        pred_valid = pred[mask].clamp_min(1e-12)
        target_valid = target[mask].clamp_min(1e-12)

        # Accumulate AbsRel and Delta1 over all valid pixels.
        absrel_sum += torch.sum(
            torch.abs(pred_valid - target_valid) / target_valid
        ).item()

        ratio = torch.maximum(
            pred_valid / target_valid,
            target_valid / pred_valid,
        )
        delta1_sum += torch.sum(ratio < 1.25).item()

        valid_pixel_count += int(mask.sum().item())

    mean_sirmse = torch.stack(per_image_sirmse).mean().item()
    absrel = absrel_sum / valid_pixel_count
    delta1 = delta1_sum / valid_pixel_count

    return {
        "model_family": MODEL_FAMILY,
        "weights_source": WEIGHTS_SOURCE,
        "checkpoint_path": str(CHECKPOINT_PATH) if WEIGHTS_SOURCE == "checkpoint" else None,
        "mean_si_RMSE": mean_sirmse,
        "AbsRel": absrel,
        "Delta1": delta1,
    }

### Run Evaluation

This cell runs the full validation loop.

In [16]:
metrics = evaluate(
    model=model,
    dataloader=val_loader,
    device=device,
)

metrics_df = pd.DataFrame([metrics])

display(
    metrics_df[
        [
            "model_family",
            "weights_source",
            "mean_si_RMSE",
            "AbsRel",
            "Delta1"
        ]
    ]
)


Evaluating:   0%|          | 0/565 [00:00<?, ?batch/s]

,model_family,weights_source,mean_si_RMSE,AbsRel,Delta1
0,DA2,checkpoint,0.551982,2.202107,0.276349


### Save Metrics

In [17]:
checkpoint = CHECKPOINT[4:-3] if WEIGHTS_SOURCE == "checkpoint" else "pretrained"
out_path = RESULTS_DIR / f"eval_{MODEL_FAMILY}_{checkpoint}.csv"

metrics_df.to_csv(out_path, index=False)

print(f"Saved metrics to: {out_path}")

Saved metrics to: ../evaluation_results/eval_DA2_finetuned.csv
